In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install torch torchvision
!pip install tensorflow
!pip install opencv-python-headless
!pip install numpy


In [3]:
# Clone the YOLOv5 repository
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5

# Check out the specific commit
!git checkout 6420a1db87460d36fd2141a65659093df27c1996

# Install dependencies
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17129, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 17129 (delta 43), reused 14 (delta 14), pack-reused 17067 (from 3)
Receiving objects: 100% (17129/17129), 15.84 MiB | 25.30 MiB/s, done.
Resolving deltas: 100% (11747/11747), done.
/content/yolov5
Note: switching to '6420a1db87460d36fd2141a65659093df27c1996'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 6420a1db Fix TFLite Segment infe

In [21]:
!pip install matplotlib

In [24]:
# Basic data processing and visualization
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

# Deep learning frameworks
import torch
import tensorflow as tf
from tensorflow.keras.models import load_model

# For displaying in Jupyter/Colab
from IPython.display import display

In [25]:
# Load the object detection model (YOLOv5)
detection_model_path = '/content/drive/MyDrive/vision/best_model.pt'
detection_model = torch.hub.load('.', 'custom', path=detection_model_path, source='local')

# Load the classification model (TensorFlow/Keras)
classification_model_path = '/content/drive/MyDrive/vision/classification_dataset/traffic_sign_classifier_model.h5'
classification_model = load_model(classification_model_path)


YOLOv5 🚀 v7.0-395-g6420a1db Python-3.10.12 torch-2.5.1+cu121 CPU

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


In [32]:
!unzip /content/test2.zip -d /content/drive/MyDrive/vision

Archive:  /content/test2.zip
   creating: /content/drive/MyDrive/vision/test2/
  inflating: /content/drive/MyDrive/vision/test2/000015_jpg.rf.1c60d15418d0291507def26c241ed27e.jpg  
  inflating: /content/drive/MyDrive/vision/test2/000041_jpg.rf.89c814e3db33e70112e3447d54bb42c6.jpg  
  inflating: /content/drive/MyDrive/vision/test2/00014_00024_00018_png.rf.1b7a484113cbc1f29d56a8f50156948a.jpg  
  inflating: /content/drive/MyDrive/vision/test2/000163_jpg.rf.55c9537881a01c071eb6c3b46008b884.jpg  
  inflating: /content/drive/MyDrive/vision/test2/000202_jpg.rf.7b9c03e897bab9b7a170deae87d31266.jpg  
  inflating: /content/drive/MyDrive/vision/test2/000359_jpg.rf.767bed64e2ff0d34ec7b1db1d0e6c398.jpg  
  inflating: /content/drive/MyDrive/vision/test2/000478_jpg.rf.ea1348c002d93944315cd4db7cccc0ed.jpg  
  inflating: /content/drive/MyDrive/vision/test2/000558_jpg.rf.40c69cee11121c32100bc0ec9dc29849.jpg  
  inflating: /content/drive/MyDrive/vision/test2/000599_jpg.rf.87f4886233a457df94743a34ee41021

In [33]:
import os
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def preprocess_for_classification(image, size=(64, 64)):
    """
    Preprocess detected object for classification model.
    Assumes the classification model expects 64x64 images (adjust size as needed).
    """
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    # Resize to expected input size
    image = image.resize(size)

    # Convert to numpy array and normalize
    image = np.array(image) / 255.0

    # Add batch dimension
    image = np.expand_dims(image, axis=0)

    return image

def detect_and_classify(image_path, detection_model, classification_model, conf_threshold=0.25):
    """
    Detect traffic signs and classify them using both models.
    """
    # Read image
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Perform detection
    results = detection_model(image_rgb)

    predictions = []
    detected_objects = []

    for detection in results.xyxy[0].cpu().numpy():  # Get detections for first image
        x1, y1, x2, y2, conf, cls = detection
        if conf >= conf_threshold:
            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
            detected_object = image_rgb[y1:y2, x1:x2]
            if detected_object.size == 0:
                continue

            processed_object = preprocess_for_classification(detected_object)
            class_prediction = classification_model.predict(processed_object, verbose=0)
            class_idx = np.argmax(class_prediction)
            class_conf = float(class_prediction[0][class_idx])

            predictions.append({
                'bbox': [x1, y1, x2, y2],
                'detection_conf': float(conf),
                'class_idx': int(class_idx),
                'class_conf': class_conf
            })
            detected_objects.append(detected_object)

            print(f"Detected object at coordinates ({x1}, {y1}, {x2}, {y2})")
            print(f"Class: {class_idx}, Confidence: {class_conf:.2f}")
            print("-" * 50)

    return predictions, detected_objects, image_rgb

def draw_predictions(image_path, detection_model, classification_model, class_names=None, save_path=None):
    """
    Draw and display predictions on the image, optionally save the result.
    """
    predictions, _, image_rgb = detect_and_classify(
        image_path, detection_model, classification_model
    )

    image_vis = image_rgb.copy()

    for pred in predictions:
        x1, y1, x2, y2 = pred['bbox']
        cv2.rectangle(image_vis, (x1, y1), (x2, y2), (0, 255, 0), 3)

        if class_names is not None:
            label = f"{class_names[pred['class_idx']]} ({pred['class_conf']:.2f})"
        else:
            label = f"Class {pred['class_idx']} ({pred['class_conf']:.2f})"

        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(image_vis, (x1, y1-25), (x1 + w, y1), (0, 255, 0), -1)
        cv2.putText(image_vis, label, (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    plt.figure(figsize=(15, 10))
    plt.imshow(image_vis)
    plt.axis('off')
    plt.title('Detected Traffic Signs with Classifications')
    plt.show()

    if save_path:
        save_img = cv2.cvtColor(image_vis, cv2.COLOR_RGB2BGR)
        cv2.imwrite(save_path, save_img)
        print(f"Saved annotated image to: {save_path}")

def process_images_in_folder(folder_path, detection_model, classification_model, class_names, output_folder):
    """
    Process all images in a folder, classify detected objects, and save the results.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Iterate through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(folder_path, filename)
            save_path = os.path.join(output_folder, f"output_{filename}")

            print(f"Processing image: {filename}")
            draw_predictions(image_path, detection_model, classification_model, class_names, save_path)

# Example usage
folder_path = '/content/drive/MyDrive/vision/test2'
output_folder = '/content/output_folder'

class_names = ['speedLimit_20', 'speedLimit_30', 'speedLimit_50',
               'speedLimit_60', 'speedLimit_70', 'speedLimit_80',
               'speedLimit_100', 'speedLimit_120', 'Stop']

process_images_in_folder(
    folder_path,
    detection_model,
    classification_model,
    class_names=class_names,
    output_folder=output_folder
)


Processing image: 000015_jpg.rf.1c60d15418d0291507def26c241ed27e.jpg


/content/yolov5/./models/common.py:895: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Detected object at coordinates (23, 55, 188, 250)
Class: 4, Confidence: 0.81
--------------------------------------------------
Saved annotated image to: /content/output_folder/output_000015_jpg.rf.1c60d15418d0291507def26c241ed27e.jpg
Processing image: 000041_jpg.rf.89c814e3db33e70112e3447d54bb42c6.jpg


/content/yolov5/./models/common.py:895: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Detected object at coordinates (169, 34, 279, 227)
Class: 1, Confidence: 1.00
--------------------------------------------------
Saved annotated image to: /content/output_folder/output_000041_jpg.rf.89c814e3db33e70112e3447d54bb42c6.jpg
Processing image: 00014_00024_00018_png.rf.1b7a484113cbc1f29d56a8f50156948a.jpg


...

Detected object at coordinates (147, 13, 279, 272)
Class: 0, Confidence: 0.89
--------------------------------------------------
Saved annotated image to: /content/output_folder/output_001482_jpg.rf.c040f508e60b3a870f5f5fc8b9b017d2.jpg
Processing image: road84_png.rf.3c4a24183c70e3af08fc01ae65761c88.jpg


/content/yolov5/./models/common.py:895: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Detected object at coordinates (153, 88, 259, 160)
Class: 8, Confidence: 1.00
--------------------------------------------------
Saved annotated image to: /content/output_folder/output_road84_png.rf.3c4a24183c70e3af08fc01ae65761c88.jpg
Processing image: road85_png.rf.86fab2367257fba23a95cad102a7dd82.jpg


/content/yolov5/./models/common.py:895: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Detected object at coordinates (118, 151, 317, 300)
Class: 8, Confidence: 1.00
--------------------------------------------------
Saved annotated image to: /content/output_folder/output_road85_png.rf.86fab2367257fba23a95cad102a7dd82.jpg
